The Taxi Data that is being worked with was downloaded from https://data.cityofchicago.org/Transportation/Taxi-Trips-2024-/ajtu-isnz/about_data.

The dataset consists of 16.1 million rows and 23 columns. Each row is a trip of a taxi in Chigago, with *Trip ID* being its unique identifier.

This full description of the dataset is copied from the website.

| Column Name | Description | Data Type |
|-------------|----------------------------|-----------|
| Trip ID | A unique identifier for the trip. | Text |
| Taxi ID | A unique identifier for the taxi. | Text |
| Trip Start Timestamp | When the trip started, rounded to the nearest 15 minutes. | Floating Timestamp |
| Trip End Timestamp | When the trip ended, rounded to the nearest 15 minutes. | Floating Timestamp |
| Trip Seconds | Time of the trip in seconds. | Number |
| Trip Miles | Distance of the trip in miles. | Number |
| Pickup Census Tract | The Census Tract where the trip began. For privacy, this Census Tract is not shown for some trips. This column often will be blank for locations outside Chicago. | Text |
| Dropoff Census Tract | The Census Tract where the trip ended. For privacy, this Census Tract is not shown for some trips. This column often will be blank for locations outside Chicago. | Text |
| Pickup Community Area | The Community Area where the trip began. This column will be blank for locations outside Chicago. | Number |
| Dropoff Community Area | The Community Area where the trip ended. This column will be blank for locations outside Chicago. | Number |
| Fare | The fare for the trip. | Number |
| Tips | The tip for the trip. Cash tips generally will not be recorded. | Number |
| Tolls | The tolls for the trip. | Number |
| Extras | Extra charges for the trip. | Number |
| Trip Total | Total cost of the trip, the total of the previous columns. | Number |
| Payment Type | Type of payment for the trip. | Text |
| Company | The taxi company. | Text |
| Pickup Centroid Latitude | The latitude of the center of the pickup census tract or the community area if the census tract has been hidden for privacy. This column often will be blank for locations outside Chicago. | Number |
| Pickup Centroid Longitude | The longitude of the center of the pickup census tract or the community area if the census tract has been hidden for privacy. This column often will be blank for locations outside Chicago. | Number |
| Pickup Centroid Location | The location of the center of the pickup census tract or the community area if the census tract has been hidden for privacy. This column often will be blank for locations outside Chicago. | Point |
| Dropoff Centroid Latitude | The latitude of the center of the dropoff census tract or the community area if the census tract has been hidden for privacy. This column often will be blank for locations outside Chicago. | Number |
| Dropoff Centroid Longitude | The longitude of the center of the dropoff census tract or the community area if the census tract has been hidden for privacy. This column often will be blank for locations outside Chicago. | Number |
| Dropoff Centroid Location | The location of the center of the dropoff census tract or the community area if the census tract has been hidden for privacy. This column often will be blank for locations outside Chicago. | Point |


In [39]:
import pandas as pd
import numpy as np
import h3

In [40]:
taxi_data_v1 = pd.read_csv("../data/Taxi_Trips_big.csv")
taxi_data_v1

KeyboardInterrupt: 

In [ ]:
taxi_data_v1.info()

<class 'pandas.DataFrame'>
RangeIndex: 16086255 entries, 0 to 16086254
Data columns (total 23 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   Trip ID                     str    
 1   Taxi ID                     str    
 2   Trip Start Timestamp        str    
 3   Trip End Timestamp          str    
 4   Trip Seconds                float64
 5   Trip Miles                  str    
 6   Pickup Census Tract         float64
 7   Dropoff Census Tract        float64
 8   Pickup Community Area       float64
 9   Dropoff Community Area      float64
 10  Fare                        str    
 11  Tips                        str    
 12  Tolls                       str    
 13  Extras                      str    
 14  Trip Total                  str    
 15  Payment Type                str    
 16  Company                     str    
 17  Pickup Centroid Latitude    str    
 18  Pickup Centroid Longitude   str    
 19  Pickup Centroid Location    st

## 1. Delete unneccessary columns
As the features *Fare*, *Tips*, *Tolls*, *Extras* and *Payment Type* are not used for analysis, they are deleted from the dataset.
The features *Pickup Centroid Location* and *Dropoff Centroid Location* are redundant and therefore deleted as well.

In [ ]:
cols_to_drop = ["Fare", "Tips", "Tolls", "Extras", "Payment Type", 
                "Pickup Centroid Location", "Dropoff Centroid  Location"]

taxi_data_v1 = taxi_data_v1.drop(columns=cols_to_drop)

<class 'pandas.DataFrame'>
RangeIndex: 16086255 entries, 0 to 16086254
Data columns (total 16 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   Trip ID                     str    
 1   Taxi ID                     str    
 2   Trip Start Timestamp        str    
 3   Trip End Timestamp          str    
 4   Trip Seconds                float64
 5   Trip Miles                  str    
 6   Pickup Census Tract         float64
 7   Dropoff Census Tract        float64
 8   Pickup Community Area       float64
 9   Dropoff Community Area      float64
 10  Trip Total                  str    
 11  Company                     str    
 12  Pickup Centroid Latitude    str    
 13  Pickup Centroid Longitude   str    
 14  Dropoff Centroid Latitude   str    
 15  Dropoff Centroid Longitude  str    
dtypes: float64(5), str(11)
memory usage: 6.2 GB


## 2. Change Data types from string to numeric/datetime
While downloading the data, some of the data types were changed to *string* and have to be changed back for Analysis.

In [ ]:
# Columns to fix data type
cols_to_fix = [
    'Pickup Centroid Longitude', 'Pickup Centroid Latitude',
    'Dropoff Centroid Longitude', 'Dropoff Centroid Latitude',
    'Trip Total', 'Trip Miles'
]

for col in cols_to_fix:
    
    if col == 'Trip Total':
        # Remove the dollar sign and commas from the Trip Total column
        taxi_data_v1[col] = taxi_data_v1[col].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False)
    elif col == 'Trip Miles' or col in ['Pickup Centroid Longitude', 'Pickup Centroid Latitude', 'Dropoff Centroid Longitude', 'Dropoff Centroid Latitude']:
        # Replace the comma with a standard decimal point
        taxi_data_v1[col] = taxi_data_v1[col].astype(str).str.replace(',', '.', regex=False)

    # Convert to numeric
    taxi_data_v1[col] = pd.to_numeric(taxi_data_v1[col], errors='coerce')

#convert to datetime
taxi_data_v1["Trip Start Timestamp"] = pd.to_datetime(
    taxi_data_v1["Trip Start Timestamp"]
)

taxi_data_v1["Trip End Timestamp"] = pd.to_datetime(
    taxi_data_v1["Trip End Timestamp"]
)

In [ ]:
taxi_data_v1.info()

<class 'pandas.DataFrame'>
RangeIndex: 16086255 entries, 0 to 16086254
Data columns (total 16 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   Trip ID                     str    
 1   Taxi ID                     str    
 2   Trip Start Timestamp        str    
 3   Trip End Timestamp          str    
 4   Trip Seconds                float64
 5   Trip Miles                  float64
 6   Pickup Census Tract         float64
 7   Dropoff Census Tract        float64
 8   Pickup Community Area       float64
 9   Dropoff Community Area      float64
 10  Trip Total                  float64
 11  Company                     str    
 12  Pickup Centroid Latitude    float64
 13  Pickup Centroid Longitude   float64
 14  Dropoff Centroid Latitude   float64
 15  Dropoff Centroid Longitude  float64
dtypes: float64(11), str(5)
memory usage: 5.4 GB


## 3. Null Value Analysis
As Null values are difficult to handle during Analysis, they have to be dealt with beforehand.

In [ ]:
# Find Null Values and add percentage of NaN values for each column
is_na_df = taxi_data_v1.isna().sum()
is_na_df = pd.DataFrame(is_na_df, columns=['NaN Count'])
is_na_df['Total Count'] = len(taxi_data_v1)
is_na_df['NaN Percentage'] = (is_na_df['NaN Count'] / is_na_df['Total Count']) * 100

# Check for any NaN values in all columns
print("NaN values before conversion:")
print(is_na_df)

NaN values before conversion:
                            NaN Count  Total Count  NaN Percentage
Trip ID                             0     16086255        0.000000
Taxi ID                            12     16086255        0.000075
Trip Start Timestamp                0     16086255        0.000000
Trip End Timestamp                427     16086255        0.002654
Trip Seconds                     3081     16086255        0.019153
Trip Miles                        157     16086255        0.000976
Pickup Census Tract           8891026     16086255       55.270950
Dropoff Census Tract          9102541     16086255       56.585831
Pickup Community Area          445622     16086255        2.770204
Dropoff Community Area        1414559     16086255        8.793588
Trip Total                      32557     16086255        0.202390
Company                             0     16086255        0.000000
Pickup Centroid Latitude       437472     16086255        2.719539
Pickup Centroid Longitude      4

As the columns *Taxi_ID*, *Trip End Timestamp*, *Trip Seconds*, *Trip Miles* and *Trip Total* are needed for the analysis, and the number of null values are relatively small (less than 0.02%), those datapoints are simply deleted from the dataset.

In [ ]:
taxi_data_v2 = taxi_data_v1.dropna(subset=['Taxi ID', 'Trip End Timestamp', 
                                                    'Trip Seconds', 'Trip Miles',
                                                    'Trip Total'])
taxi_data_v2.isna().sum()

NameError: name 'taxi_data_v1' is not defined

For the spatial entities the number of Null values is relatively big, especially for *Census Tract*, so further analysis is neccessary.

In [ ]:
spatial_cols = [
    "Pickup Census Tract",
    "Dropoff Census Tract",
    "Pickup Community Area",
    "Dropoff Community Area",
    "Pickup Centroid Latitude",
    "Pickup Centroid Longitude",
    "Dropoff Centroid Latitude",
    "Dropoff Centroid Longitude",
]
null_mask = taxi_data_v2[spatial_cols].isna()

conditional_overlap = pd.DataFrame(
    index=spatial_cols,
    columns=spatial_cols,
    dtype=float
)

for c1 in spatial_cols:
    missing_c1 = null_mask[c1].sum()

    for c2 in spatial_cols:
        if missing_c1 == 0:
            conditional_overlap.loc[c1, c2] = 0
        else:
            conditional_overlap.loc[c1, c2] = (
                (null_mask[c1] & null_mask[c2]).sum() / missing_c1
            )

conditional_overlap = conditional_overlap.round(3)

conditional_overlap

,Pickup Census Tract,Dropoff Census Tract,Pickup Community Area,Dropoff Community Area,Pickup Centroid Latitude,Pickup Centroid Longitude,Dropoff Centroid Latitude,Dropoff Centroid Longitude
Pickup Census Tract,1.000,0.997,0.049,0.121,0.049,0.049,0.121,0.121
Dropoff Census Tract,0.974,1.000,0.045,0.144,0.045,0.045,0.144,0.144
Pickup Community Area,0.977,0.911,1.000,0.671,0.982,0.982,0.655,0.655
Dropoff Community Area,0.759,0.929,0.211,1.000,0.207,0.207,0.941,0.941
Pickup Centroid Latitude,0.995,0.927,1.000,0.668,1.000,1.000,0.667,0.667
Pickup Centroid Longitude,0.995,0.927,1.000,0.668,1.000,1.000,0.667,0.667
Dropoff Centroid Latitude,0.806,0.987,0.220,1.000,0.219,0.219,1.000,1.000
Dropoff Centroid Longitude,0.806,0.987,0.220,1.000,0.219,0.219,1.000,1.000


The overlap matrix shows that data points with missing values in the *Pickup/Dropoff Community Area* fields almost always also have missing values in the corresponding *Census Tract* and *Latitude/Longitude* fields. As these observations do not contain sufficient spatial information for either spatial aggregation, they are removed from the dataset.

More than half of the observations have missing values for the *Census Tract*. This is expected, as Census Tracts are omitted for privacy reasons in some trips and are also unavailable for locations outside the city of Chicago.

Since the *Community Area* and *Latitude/Longitude* values are available for most of these observations, the dataset is split after the data preparation step. For the Census Tract analysis, a dataset containing only observations with available Census Tracts is used. For the Community Area analysis, a larger dataset is retained in which the Census Tract may be missing.

In [ ]:
taxi_data_v3 = taxi_data_v2.dropna(subset=['Pickup Community Area', 'Dropoff Community Area'])
taxi_data_v3.isna().sum()

## 4. Add H3 Index Columns

For the subsequent spatial analyses, additional spatial entities are added to the dataset by computing H3 indices at resolutions 7, 8, and 9.

The H3 indices are derived from the available pickup and dropoff latitude and longitude coordinates. Due to privacy restrictions, these coordinates do not represent the exact pickup or dropoff locations. Instead, they correspond to the centroid of the associated Census Tract. If the Census Tract is unavailable, the centroid of the corresponding Community Area is provided instead.

Consequently, the H3 indices do not introduce additional spatial information. Instead, they provide an alternative spatial discretization of the city at different levels of granularity, which is used in the subsequent descriptive spatial analysis.

In [ ]:
taxi_data_v3['h3_index_pickup_7'] = [
    # Check if lat and lng are not NaN before converting to H3 index, otherwise return None
    h3.latlng_to_cell(lat, lng, 7) if (lat == lat and lng == lng) else None
    for lat, lng in zip(
        taxi_data_v3['Pickup Centroid Latitude'], 
        taxi_data_v3['Pickup Centroid Longitude']
    )
]
taxi_data_v3['h3_index_dropoff_7'] = [
    # Check if lat and lng are not NaN before converting to H3 index, otherwise return None
    h3.latlng_to_cell(lat, lng, 7) if (lat == lat and lng == lng) else None
    for lat, lng in zip(
        taxi_data_v3['Dropoff Centroid Latitude'], 
        taxi_data_v3['Dropoff Centroid Longitude']
    )
]

taxi_data_v3['h3_index_pickup_8'] = [
    # Check if lat and lng are not NaN before converting to H3 index, otherwise return None
    h3.latlng_to_cell(lat, lng, 8) if (lat == lat and lng == lng) else None
    for lat, lng in zip(
        taxi_data_v3['Pickup Centroid Latitude'], 
        taxi_data_v3['Pickup Centroid Longitude']
    )
]
taxi_data_v3['h3_index_dropoff_8'] = [
    # Check if lat and lng are not NaN before converting to H3 index, otherwise return None
    h3.latlng_to_cell(lat, lng, 8) if (lat == lat and lng == lng) else None
    for lat, lng in zip(
        taxi_data_v3['Dropoff Centroid Latitude'], 
        taxi_data_v3['Dropoff Centroid Longitude']
    )
]

taxi_data_v3['h3_index_pickup_9'] = [
    # Check if lat and lng are not NaN before converting to H3 index, otherwise return None
    h3.latlng_to_cell(lat, lng, 9) if (lat == lat and lng == lng) else None
    for lat, lng in zip(
        taxi_data_v3['Pickup Centroid Latitude'], 
        taxi_data_v3['Pickup Centroid Longitude']
    )
]
taxi_data_v3['h3_index_dropoff_9'] = [
    # Check if lat and lng are not NaN before converting to H3 index, otherwise return None
    h3.latlng_to_cell(lat, lng, 9) if (lat == lat and lng == lng) else None
    for lat, lng in zip(
        taxi_data_v3['Dropoff Centroid Latitude'], 
        taxi_data_v3['Dropoff Centroid Longitude']
    )
]

In [ ]:
# Check for any NaN values in the new H3 index columns
print(taxi_data_v3[['h3_index_pickup_7', 
                 'h3_index_dropoff_7',
                 'h3_index_pickup_8',
                 'h3_index_dropoff_8',
                 'h3_index_pickup_9',
                 'h3_index_dropoff_9',
                 'Pickup Census Tract', 
                 'Dropoff Census Tract']].isna().sum())

columns_to_drop = [
    'Pickup Centroid Longitude', 'Pickup Centroid Latitude',
    'Dropoff Centroid Longitude', 'Dropoff Centroid Latitude'
]

#TODO: ask if anybody still needs Long/lat values else drop them 

h3_index_pickup_7             0
h3_index_dropoff_7            0
h3_index_pickup_8             0
h3_index_dropoff_8            0
h3_index_pickup_9             0
h3_index_dropoff_9            0
Pickup Census Tract     7659353
Dropoff Census Tract    7659353
dtype: int64


## 5. Outlier Analysis

The numerical variables are inspected for implausible values. Negative trip durations, trip distances and trip costs would indicate data quality issues and are therefore checked. No additional filtering based on statistical outliers is performed druing data preparation, as extreme values may correspond to legitimate taxi trips (e.g., airport transfers or unusually long trips) and therefore contain relevant information for the subsequent analyses.

In [ ]:
taxi_data_v3 = taxi_data_v3[
    (taxi_data_v3["Trip Seconds"] > 0) &
    (taxi_data_v3["Trip Miles"] >= 0) &
    (taxi_data_v3["Trip Total"] >= 0)
]

## 6. Save the dataset as parquet for all further Analysis
The big dataset *taxi_data_processed_big* is being used for Communinty Area Analysis.

In [ ]:
taxi_data_v3.to_parquet(
    "../data/processed/taxi_data_processed_big.parquet"
)

The small dataset *taxi_data_processed_small* is being used for Census Tract Analysis.

In [ ]:
taxi_data_small = taxi_data_v3.dropna(subset=['Pickup Census Tract', 'Dropoff Census Tract'])

In [ ]:
taxi_data_small.to_parquet(
    "../data/processed/taxi_data_processed_small.parquet"
)